# 🤖 CIFAR-10 Image Classification with a CNN for the e-puck Robot

**Applied Robotics and AI Project, Part 2 · Assignment 3** · BEng (Hons) Robotics and Artificial Intelligence, University of Hertfordshire

A convolutional neural network (CNN) is trained on the CIFAR-10 dataset **off the robot** (on a Colab GPU). The trained model is then tested on my own photos and on an image from the **e-puck camera in Webots**, and finally made **smaller and faster** so that it can run on the robot without disturbing its movement.

| Step | Content | Assignment requirement |
|:-:|---|---|
| 1 | Setup | |
| 2 | Load the CIFAR-10 dataset | |
| 3 | Build the CNN | **Show all the layers of the CNN** |
| 4 | Train the CNN | |
| 5 | Test accuracy and error analysis | **Show the accuracy score** |
| 6 | Predict my own images and the Webots camera image | **Predictions on own images + e-puck camera image** |
| 7 | Optimise the model for the robot | **Optimisation proposal** |
| 8 | Save and download the results | Model file for Webots, figures for the report |
| 9 | Summary, key findings and future work | |

### ▶ How to run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all** (about 5–7 minutes on a T4 GPU)
3. At step 6, click **Choose Files** to test your own images, or **Cancel upload** to use the saved images (my car, cat and bird photos and four Webots e-puck camera images: dog, deer, horse and truck).

Every cell prints what it is doing (**▶ start**, **•** progress, **✔ done** with the time taken), so you can follow the run in the output.

---
## Background and motivation

A mobile robot such as the **e-puck** sees the world through a small colour camera (52 × 39 pixels in Webots). In the first part of the project the robot recognised objects only by **colour** (red, green and blue blocks). Colour alone cannot tell a dog from a deer or a car from a truck. A **CNN** learns shapes and textures from thousands of labelled images, so it can recognise *what* an object is.

**CIFAR-10** is a standard benchmark with 10 everyday classes (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck) and small 32 × 32 images, a similar resolution to the e-puck camera. This makes it a good first step towards object recognition on the robot.

Training needs much more computing power than a small robot has, so the CNN is **trained here on a GPU** and only the trained weights are copied to the robot, which runs the much cheaper forward pass.

---
## 1. Setup
Imports, random seed and the device (GPU if available).

In [ ]:
#@title 1.1 Install packages and import libraries
print("▶ Step 1.1  Installing torchinfo and datasets...")
!pip -q install torchinfo datasets
print("✔ Packages installed")

import copy, io, json, os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torchinfo import summary
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

from contextlib import contextmanager

from IPython.display import HTML, display

def header(title):
    """Print a boxed heading for each step, so every run is easy to follow in the output."""
    line = "─" * (len(title) + 6)
    print(f"\n┌{line}┐\n│ ▶  {title}  │\n└{line}┘")

@contextmanager
def step(title):
    """Boxed heading at the start, and the time taken at the end."""
    header(title)
    start = time.time()
    yield
    print(f"✔ Done in {time.time() - start:.1f} s")

def info(message):
    print(f"   • {message}")

def show_table(data, title=None, fmt=None, bar=None, highlight=None, index=False):
    """Show a table in a neat box: title bar, coloured header row, striped rows."""
    df = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    st = df.style
    if not index:
        st = st.hide(axis="index")
    if fmt:
        st = st.format(fmt)
    if bar:
        st = st.bar(subset=bar, color="#8FB8DE", vmin=0, vmax=100)
    if highlight:
        st = st.highlight_min(subset=highlight, color="#DDF0DA")
    numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric:
        st = st.set_properties(subset=numeric, **{"text-align": "right"})
    st = st.set_table_styles([
        {"selector": "", "props": "border-collapse: collapse; margin: 0; background: #FFFFFF; color: #1A1A1A;"},
        {"selector": "th.col_heading", "props": "background: #1F4E79; color: #FFFFFF; font-weight: 600; "
                                                "padding: 8px 14px; text-align: left; border: none;"},
        {"selector": "th.row_heading", "props": "background: #EEF3F9; color: #1F4E79; font-weight: 600; padding: 6px 12px;"},
        {"selector": "th.blank", "props": "background: #1F4E79; border: none;"},
        {"selector": "td", "props": "padding: 6px 14px; border-top: 1px solid #E3E8EF; color: #1A1A1A;"},
        {"selector": "tbody tr", "props": "background: #FFFFFF;"},
        {"selector": "tbody tr:nth-child(even)", "props": "background: #F6F9FC;"},
    ])
    caption = (f'<div style="background:#EEF3F9;color:#1F4E79;font-weight:700;padding:8px 14px;'
               f'border-bottom:2px solid #1F4E79">{title}</div>') if title else ""
    display(HTML(f'<div style="display:inline-block;border:1px solid #C9D3E0;border-radius:8px;overflow:hidden;'
                 f'margin:6px 0 12px;box-shadow:0 1px 4px rgba(0,0,0,.10);font-family:Arial,Helvetica,sans-serif;'
                 f'font-size:13px;background:#FFFFFF">{caption}{st.to_html()}</div>'))

warnings.filterwarnings("ignore")
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.family": "serif", "axes.spines.top": False, "axes.spines.right": False})
NAVY, BLUE, GREEN, AMBER, RED = "#1F4E79", "#8FB8DE", "#5FA857", "#F6C453", "#C0392B"

print("✔ Libraries imported")
info(f"PyTorch {torch.__version__}, torchvision {torchvision.__version__}")
info(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda"
                             else "  ⚠ no GPU: choose Runtime → Change runtime type → T4 GPU"))
info(f"Figures and tables are saved in the '{OUT}/' folder")

The model is saved as `cnn_model.py`. The Webots controller imports this same file, so the network trained here is exactly the one that runs on the robot.

In [ ]:
%%writefile cnn_model.py
"""CNN for CIFAR-10 (32 x 32 RGB images, 10 classes).

The same file is used by the training notebook and by the Webots controller,
so the network that is trained is exactly the network that runs on the robot.
"""
import torch
import torch.nn as nn

CLASSES = ("airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck")

# Per-channel mean and standard deviation of the CIFAR-10 training set.
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


def conv_block(c_in, c_out, dropout):
    """Two 3x3 convolutions (each followed by batch norm and ReLU), then 2x2 max pooling."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),          # halves the height and width
        nn.Dropout(dropout),
    )


class CifarCNN(nn.Module):
    """VGG-style network: three convolution blocks and a two-layer classifier.

    Input 3 x 32 x 32 -> block 1 (32 ch, 16 x 16) -> block 2 (64 ch, 8 x 8)
    -> block 3 (128 ch, 4 x 4) -> fully connected 256 -> 10 class scores.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32, 0.2),
            conv_block(32, 64, 0.3),
            conv_block(64, 128, 0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def preprocess(rgb_uint8):
    """Turn an H x W x 3 uint8 RGB tensor into a normalised 1 x 3 x 32 x 32 batch.

    The largest centred square is cropped first, so the object is not squashed,
    and then resized to 32 x 32 with antialiasing.
    """
    img = rgb_uint8.permute(2, 0, 1).float().div(255.0)          # 3 x H x W, range 0..1
    _, h, w = img.shape
    s = min(h, w)
    top, left = (h - s) // 2, (w - s) // 2
    img = img[:, top:top + s, left:left + s]
    img = nn.functional.interpolate(img.unsqueeze(0), size=(32, 32), mode="bilinear",
                                    align_corners=False, antialias=True)
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    return (img - mean) / std

In [ ]:
import importlib, cnn_model
importlib.reload(cnn_model)
from cnn_model import CifarCNN, CLASSES, CIFAR_MEAN, CIFAR_STD, preprocess
print("✔ CNN definition loaded from cnn_model.py")
info("Classes: " + ", ".join(CLASSES))

---
## 2. Load CIFAR-10

60,000 colour images of 32 × 32 pixels in 10 classes: **50,000 for training** and **10,000 for testing**.

* **Augmentation** (training only): random crop with 4-pixel padding and horizontal flip, so the model does not just memorise the images.
* **Normalisation**: each colour channel is scaled with the CIFAR-10 mean and standard deviation.

The data is loaded from the Hugging Face mirror (fast). If that fails, it falls back to the official site.

**Speed:** all 60,000 images are stored once in GPU memory (about 180 MB), and the random crop and flip are done on the GPU for a whole batch at once. Colab has only 2 CPU cores, so preparing images on the CPU would make the GPU wait; this way one epoch takes a few seconds instead of about 20.

In [ ]:
class ImageList:
    """The original PIL images and labels (used for showing pictures)."""
    def __init__(self, images, labels):
        self.images, self.targets = images, list(labels)
    def __len__(self):
        return len(self.targets)

MEAN_T = torch.tensor(CIFAR_MEAN, device=DEVICE).view(1, 3, 1, 1)
STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)

def to_tensor(images):
    """PIL images -> one uint8 tensor (N, 3, 32, 32) on the GPU."""
    arr = np.stack([np.asarray(im, dtype=np.uint8) for im in images])
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous().to(DEVICE)

def random_crop_flip(x):
    """Random 32x32 crop from a 4-pixel zero-padded image and random horizontal flip, for a whole batch on the GPU."""
    n, c = x.shape[:2]
    xp = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n,), device=x.device)
    j = torch.randint(0, 9, (n,), device=x.device)
    ar = torch.arange(32, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None]
    cols = (j[:, None] + ar)[:, None, None, :]
    out = xp[torch.arange(n, device=x.device)[:, None, None, None], torch.arange(c, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(n, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)

class GPULoader:
    """Batches straight from GPU memory: normalised, with optional augmentation."""
    def __init__(self, x_uint8, y, batch_size, shuffle=False, augment=False):
        self.x, self.y, self.bs, self.shuffle, self.augment = x_uint8, y, batch_size, shuffle, augment
    def __len__(self):
        return (len(self.y) + self.bs - 1) // self.bs
    def __iter__(self):
        n = len(self.y)
        order = torch.randperm(n, device=self.x.device) if self.shuffle else torch.arange(n, device=self.x.device)
        for k in range(0, n, self.bs):
            idx = order[k:k + self.bs]
            xb = self.x[idx].float().div_(255)
            if self.augment:
                xb = random_crop_flip(xb)
            yield ((xb - MEAN_T) / STD_T).contiguous(memory_format=torch.channels_last), self.y[idx]

from tqdm.auto import tqdm

def load_cifar10():
    try:                                                  # fast mirror
        info("Downloading from the Hugging Face mirror (uoft-cs/cifar10)...")
        from datasets import load_dataset
        ds = load_dataset("uoft-cs/cifar10")
        sets = {}
        for split in ("train", "test"):
            imgs = [im.convert("RGB") for im in tqdm(ds[split]["img"], desc=f"   decoding {split} images", unit="img", mininterval=1)]
            sets[split] = (imgs, ds[split]["label"])
    except Exception as err:                              # official site (slower)
        info(f"Mirror not available ({type(err).__name__}); downloading from the official CIFAR-10 site...")
        sets = {}
        for split, train in (("train", True), ("test", False)):
            raw = torchvision.datasets.CIFAR10("data", train=train, download=True)
            sets[split] = ([raw[i][0] for i in tqdm(range(len(raw)), desc=f"   preparing {split} images")], raw.targets)
    return sets

with step("Step 2.1  Loading CIFAR-10"):
    sets = load_cifar10()
    train_set = ImageList(*sets["train"])
    test_set = ImageList(*sets["test"])
    info(f"Copying all images to {DEVICE.type.upper()} memory...")
    X_train, y_train = to_tensor(train_set.images), torch.tensor(train_set.targets, device=DEVICE)
    train_set.images = None                                # the training pictures are not needed any more (frees RAM)
    X_test, y_test = to_tensor(test_set.images), torch.tensor(test_set.targets, device=DEVICE)
    train_loader = GPULoader(X_train, y_train, 128, shuffle=True, augment=True)
    test_loader = GPULoader(X_test, y_test, 1000)
    info(f"{len(train_set):,} training images in {len(train_loader)} batches of 128 (augmented on the {DEVICE.type.upper()})")
    info(f"{len(test_set):,} test images in batches of 1,000")

show_table(pd.DataFrame({"Split": ["Training", "Test"], "Images": [len(train_set), len(test_set)],
                         "Size": ["32 × 32 × 3"] * 2, "Classes": [10, 10]}),
           "CIFAR-10 dataset", fmt={"Images": "{:,}"})

In [ ]:
header("Step 2.2  Showing two example test images per class")
fig, axes = plt.subplots(2, 10, figsize=(13, 3))
for c, name in enumerate(CLASSES):
    idx = [i for i, y in enumerate(test_set.targets) if y == c][:2]
    for r in range(2):
        axes[r, c].imshow(test_set.images[idx[r]]); axes[r, c].axis("off")
    axes[0, c].set_title(name, fontsize=9)
plt.suptitle("CIFAR-10: two test images per class", y=1.02)
plt.savefig(f"{OUT}/cifar10_samples.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/cifar10_samples.png")

---
## 3. Build the CNN: all layers

A VGG-style network: **three convolution blocks**, then a **classifier**.

| Stage | Layers | Output size |
|---|---|---|
| Input | RGB image | 3 × 32 × 32 |
| Block 1 | 2 × (Conv 3×3, 32 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.2 | 32 × 16 × 16 |
| Block 2 | 2 × (Conv 3×3, 64 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.3 | 64 × 8 × 8 |
| Block 3 | 2 × (Conv 3×3, 128 filters → BatchNorm → ReLU) → MaxPool → Dropout 0.4 | 128 × 4 × 4 |
| Classifier | Flatten → Dense 256 → BatchNorm → ReLU → Dropout 0.5 → Dense 10 | 10 class scores |

In [ ]:
header("Step 3.1  Building the CNN and printing its summary")
model = CifarCNN().to(DEVICE).to(memory_format=torch.channels_last)   # channels-last: faster convolutions on the GPU
print(summary(model, input_size=(1, 3, 32, 32), col_names=("output_size", "num_params"),
              depth=3, device=DEVICE, verbose=0))
print("✔ CNN built on", DEVICE)

In [ ]:
header("Step 3.2  Passing a test input through every layer to list its output shape")
rows, x = [], torch.zeros(1, 3, 32, 32, device=DEVICE)
model.eval()
with torch.no_grad():
    for name, layer in model.named_modules():
        if len(list(layer.children())) == 0:
            x = layer(x)
            rows.append({"Layer": name, "Type": layer.__class__.__name__,
                         "Output shape": " × ".join(map(str, x.shape[1:])),
                         "Parameters": sum(p.numel() for p in layer.parameters())})
layers_df = pd.DataFrame(rows); layers_df.index += 1
TOTAL_PARAMS = int(layers_df["Parameters"].sum())
layers_df.to_csv(f"{OUT}/model_layers.csv")
show_table(layers_df, "All layers of the CNN", fmt={"Parameters": "{:,}"}, index=True)
print(f"✔ {len(layers_df)} layers, {TOTAL_PARAMS:,} parameters ({TOTAL_PARAMS * 4 / 1e6:.2f} MB as 32-bit numbers)")
info(f"Saved {OUT}/model_layers.csv")

---
## 4. Train

| Setting | Value |
|---|---|
| Optimiser | SGD, momentum 0.9 (Nesterov), weight decay 5 × 10⁻⁴ |
| Learning rate | One-cycle schedule, peak 0.1 |
| Loss | Cross-entropy with label smoothing 0.1 |
| Epochs / batch size | 30 / 128 (mixed precision on the GPU) |

Each epoch shows a progress bar (391 steps of 128 images) with the training accuracy and loss, then the test (validation) accuracy and loss. ★ marks the best epoch so far. With the data on the GPU, an epoch takes only a few seconds on a T4, so all 30 epochs finish in a few minutes. Use the `EPOCHS` slider for a quicker test run (fewer epochs give a lower accuracy).

In [ ]:
EPOCHS = 30  #@param {type:"slider", min:5, max:40, step:5}
torch.backends.cudnn.benchmark = True        # pick the fastest convolution algorithms for this GPU
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, nesterov=True, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, epochs=EPOCHS, steps_per_epoch=len(train_loader))
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

def evaluate(net, loader=test_loader, device=DEVICE):
    """Return (loss, accuracy) on a data loader."""
    net.eval(); correct = total = 0; loss_sum = 0.0
    with torch.inference_mode():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            if device.type == "cpu":
                xb = xb.contiguous()
            with torch.autocast(device.type, enabled=device.type == "cuda"):
                out = net(xb).float()
            loss_sum += F.cross_entropy(out, yb, reduction="sum").item()
            correct += (out.argmax(1) == yb).sum().item(); total += yb.numel()
    return loss_sum / total, correct / total

def train_epoch(net, opt, sched=None, bar=None):
    """One pass over the training set. Returns (loss, accuracy). Updates the progress bar if given."""
    net.train(); total = 0
    loss_sum = torch.zeros((), device=DEVICE); correct = torch.zeros((), device=DEVICE)   # kept on the GPU (no waiting)
    for step, (xb, yb) in enumerate(train_loader, 1):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            out = net(xb); loss = criterion(out, yb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        if sched: sched.step()
        loss_sum += loss.detach().float() * yb.numel()
        correct += (out.argmax(1) == yb).sum(); total += yb.numel()
        if bar is not None:
            bar.update(1)
            if step % 50 == 0 or step == len(train_loader):
                bar.set_description_str(f"accuracy: {correct.item()/total:.4f} - loss: {loss_sum.item()/total:.4f}")
    return loss_sum.item() / total, correct.item() / total

header(f"Step 4.1  Training for {EPOCHS} epochs on {DEVICE} ({len(train_loader)} steps per epoch)")
info("The best model (highest test accuracy) is saved to cifar10_cnn.pth")
history, best_acc, t0 = [], 0.0, time.time()
for epoch in range(1, EPOCHS + 1):
    bar = tqdm(total=len(train_loader), desc="training...", unit="step",
               bar_format=f"Epoch {epoch}/{EPOCHS}  {{n_fmt}}/{{total_fmt}} {{bar}} {{elapsed}} - {{desc}}")
    tl, ta = train_epoch(model, optimizer, scheduler, bar)
    vl, va = evaluate(model)
    history.append({"epoch": epoch, "train_loss": tl, "train_acc": ta, "test_loss": vl, "test_acc": va})
    best = va > best_acc
    if best:
        best_acc = va; torch.save(model.state_dict(), "cifar10_cnn.pth")
    bar.set_description_str(f"accuracy: {ta:.4f} - loss: {tl:.4f} - val_accuracy: {va:.4f} - val_loss: {vl:.4f}"
                            + ("  ★ best" if best else ""))
    bar.close()
TRAIN_TIME = time.time() - t0
model.load_state_dict(torch.load("cifar10_cnn.pth", map_location=DEVICE))
print(f"✔ Training finished in {TRAIN_TIME/60:.1f} min. Best test accuracy: {best_acc*100:.2f}%")

In [ ]:
header("Step 4.2  Plotting the training curves")
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h.epoch, h.train_loss, color=BLUE, lw=2, label="training")
ax[0].plot(h.epoch, h.test_loss, color=NAVY, lw=2, label="test")
ax[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
ax[1].plot(h.epoch, h.train_acc * 100, color=BLUE, lw=2, label="training")
ax[1].plot(h.epoch, h.test_acc * 100, color=NAVY, lw=2, label="test")
ax[1].set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
for a in ax: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", bbox_inches="tight"); plt.show()
gap = (h.train_acc.iloc[-1] - h.test_acc.iloc[-1]) * 100
print(f"✔ Saved {OUT}/training_curves.png")
info(f"Final training accuracy {h.train_acc.iloc[-1]*100:.1f}%, test accuracy {h.test_acc.iloc[-1]*100:.1f}% "
     f"(gap {gap:.1f} points: {'small, so little overfitting' if gap < 8 else 'some overfitting'})")

---
## 5. Test accuracy

The trained model is tested on the **10,000 test images** it has never seen.

In [ ]:
header(f"Step 5.1  Classifying the {len(test_set):,} test images")
model.eval(); preds, labels = [], []
with torch.inference_mode():
    for xb, yb in test_loader:
        preds.append(model(xb).argmax(1).cpu()); labels.append(yb.cpu())
preds, labels = torch.cat(preds).numpy(), torch.cat(labels).numpy()
TEST_ACC = float((preds == labels).mean())
cm = confusion_matrix(labels, preds)
per_class = cm.diagonal() / cm.sum(1)

print(f"✔ Test accuracy: {TEST_ACC*100:.2f}%   ({(preds == labels).sum():,} of {len(labels):,} images correct)")
show_table(pd.DataFrame({"Class": CLASSES, "Accuracy (%)": (per_class * 100).round(1)})
           .sort_values("Accuracy (%)", ascending=False),
           "Test accuracy per class", fmt={"Accuracy (%)": "{:.1f}"}, bar=["Accuracy (%)"])

In [ ]:
header("Step 5.2  Confusion matrix: which classes are mixed up")
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
ax.set_title(f"Confusion matrix (test accuracy {TEST_ACC*100:.1f}%)")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", bbox_inches="tight"); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
i, j = np.unravel_index(off.argmax(), off.shape)
print(f"✔ Saved {OUT}/confusion_matrix.png")
info(f"Easiest class: {CLASSES[per_class.argmax()]} ({per_class.max()*100:.1f}%)")
info(f"Hardest class: {CLASSES[per_class.argmin()]} ({per_class.min()*100:.1f}%)")
info(f"Most common mistake: {CLASSES[i]} predicted as {CLASSES[j]} ({off[i, j]} images)")

In [ ]:
header("Step 5.3  Examples of test images the model got wrong")
wrong = np.where(preds != labels)[0][:12]
fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for a, k in zip(axes.flat, wrong):
    a.imshow(test_set.images[k]); a.axis("off")
    a.set_title(f"true: {CLASSES[labels[k]]}\npred: {CLASSES[preds[k]]}", fontsize=8, color=RED)
plt.suptitle("Examples of mistakes"); plt.tight_layout()
plt.savefig(f"{OUT}/misclassified.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/misclassified.png")

**Comparison with other models on CIFAR-10**

In [ ]:
bench = pd.DataFrame([
    {"Model": "Random guess", "Test accuracy (%)": "10", "Parameters": "–"},
    {"Model": "Simple CNN (3 conv layers, Keras tutorial style)", "Test accuracy (%)": "≈ 70", "Parameters": "≈ 0.12 M"},
    {"Model": "This model (6 conv layers + BatchNorm + Dropout)", "Test accuracy (%)": f"{TEST_ACC*100:.1f}", "Parameters": f"{TOTAL_PARAMS/1e6:.2f} M"},
    {"Model": "ResNet-18 (He et al., 2016)", "Test accuracy (%)": "≈ 93–95", "Parameters": "11.2 M"},
])
show_table(bench, "Comparison with other models on CIFAR-10")

**Why some classes are harder.** Cat, dog, bird and deer are animals with similar shapes, colours and backgrounds, and at 32 × 32 pixels the small details that separate them (ears, snout, legs) are only a few pixels wide. Vehicles (automobile, truck, ship) have strong straight edges and typical backgrounds (road, sea), so they are easier. A deeper network such as ResNet-18 reaches about 94% but has 14 times more parameters, which is too heavy for a small robot.

---
## 6. Predict your own images

When the cell runs, a **Choose Files** button appears under it.

* Pick your images: for example **two of your own photos** and **one image from the e-puck camera** in Webots (`controllers/Assignment_3/captures/`).
* Press **Cancel upload** (or set `IMAGE_SOURCE` to *my saved images*) to use my saved images: **car, cat and bird photos** and four **Webots e-puck camera images** (dog, deer, horse and truck) from the arena. They are stored inside this notebook.
* Run this cell again to try more images; no retraining is needed.

**Object crop and test-time augmentation (TTA).** CIFAR-10 objects fill most of the 32 × 32 frame, but in the Webots images the animal is small and about half of the frame is blue sky. CIFAR-10 airplanes are usually photographed against a blue sky, so a plain centre crop pushes the CNN towards *airplane*. With `SMART_CROP` on:

1. For Webots images (recognised by the pink arena floor), the pixels that differ from the sky and hills of their own row and lie above the floor edge are taken as the object, and a square around the largest blob is cropped. Other photos keep the centred square crop.
2. The CNN classifies the crop at two zoom levels and their mirror images, and the four results are averaged (TTA).

The table shows the prediction with the plain centre crop next to the improved one, so the effect can be compared.

In [ ]:
#@title 6.1 Saved test images (embedded in the notebook, so no download is needed)
SAVED_IMAGES = {
"car.jpg": "...",
"cat.jpg": "...",
"bird.jpg": "...",
"webots_dog.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAIAAADPWPyJAAANMElEQVR4Xm1YW6wlR3Wtvauq+5w+555z3zNzBw8YY3CwMTZEmsiRIR9YEEIESEAiIWQZfiL44Au++JkfXlLygxOkKJJ/okhB4REURUQBARYWiMBIkNjYHozHxuPxvO7jvLq7HnujXdXn3GtC3zt3Tvfp7lq199prryp46CMXlAJmAmaVDvmPWUF32l3M30L6prsmx6OfWF8sagru6isHO5aI6OKlq9986R6lFOYb5dUnHgA5OXkhvRnypXydGfI5AiCAQkRCrdIvAMoBGkCDQrlHbtOA8oMA6RE5Hv2YJt9qiJef+62fHALoi0++8Oi3/ufB05cMYn5O6fS27tCoVl/kr2UgeW2+LqNojQa1Aa2N3M8CHxTnmabJnZhfilmaSjexFBV8ffjhK9fuQ1CTg4Onf3X57jv3fnzx6b/79sULn/5rVri7fe1bv9o7GaQccUiRO06RjEfLhKUhQMVlHOE9H/18l09IAe2SKg9xgtsl9A8df773i6q0h0fTycHkXx9/5kPvuOudD9xbR9MqM5/NXnt268vfsQDQEWFFkxMZTNGQ85OpX00A3vuxL3ZZz+lO4E6SL0fs/0NMTFHPPfEPd53bOv+mU6MCd3a3q7WhHYwIoKp626fPjrZvI19rrsEMyc2UimhK0CbUMz+5ziHa9dNgepH4k39/+ffiKnR638NfzoAyL1N+X4XuVQWR7pNgHFeLnF796T9+6oP374yqcjAcb68PN7er4RhNiXZE7SFiBF35el8xa1sowKPnn2aC/u65YrQNxRqDid7Nj262TTPs68/88yQPagC1pBIUU4IngRL+LfGiIuEEAHZZTo+tCJiPvT/5G2N/iWh2dsfV+kYxGETX+HqGtgYKaJipbecTYtLGuPlsPp3r3hDaNkwPQC/AVJGiUmoybz//6Lffd/71t5+qhiUarXUXLRRsXV6XwpGACCeENzm2HX+Pjxzpb7x036fufwG15hD89MC7wKZQAbTy4JWra+88KyYKi1u3QtPawIRGF1Nd9HS5TqBC4MX0SN32/i899oXS6KpfGsQuCgmigJNKEAVZEU0+M3O6tMTaqd4yjOkYjEdaYWyaJviIJkZLcWrAxxja+SL44GOIrm2n8+CpLJyezMvBcLC5bpRReuBDaNpw8NTX3v4Xn8v8MyZFbqm9xxqcIJ4QS0lqyu+rFHRZ9Km0Tb8K0UU/h2rQxmJ6VPtmYdG7pm7rpmld8F5RVCHGQCqGwXBAqG3bh9JzmDvnZ4tGcTBa5yAYbY4jl1tDrs4VsmOBP5aolOjUKzJ8AQcwb4lCrTggYeP4xv7s8OBWZeNiOmsb13rnW2cNGlBAikNwPuqyqGIsWM7apnVNc/b8IzlszGyMyVnrSjQFLKd0qSe4kuSMqSuIk90tz+axn+88fMcVaxHms0jlZF4/8YsXXrMOs+li3rh53SoiBDUojSQrhK31AferYtTqsmZdONcSkzUos06BMEYncN0/yHottUkrdKuI/uEjFTixIiT91HPXbtvsD0ahUebwcHLl+tHlFyeKIhEpIkgqECPFQDEGVgf3zsKf2sJYW5RFdA2F1lp9DM4WXeSkEASy1L28JWn2Sd4ngVuFdEVH6dKJISUbeor/8uX/e+wd97+2cdQDt79/k6OTDip35JchkWpdOLtRXTlcXPz1dVuYzfFg2Cs5ko6NtamPJlUwVmOuU0qYSCKXy1agMIEGkSD07Y1nv2vWzw3PvFnrIrNAbmun06tP0tEVokZjoS3PwtjVbbNobr0ymSw86OKhB+7TRjJBAbDaULrXLg7YHd25+Udmbfd7X//KvXec2tsYKDCFxvrZ/9DGQnRNjKYomEHr4ANK0JkRiHwzU0CoFLvJ0dUn42RWkwsqqJfmk1cuDbSPqBnAoOJYoG2UKsRDIMdACGDLXkm4P7sxXhs88Md3l70CAQ6PFpG5aA6F7MGhKsLRb/zk+fPn3/6TX15+19teNxoOrLa3b/CLRzUBVtaYZx7/lxs3JpEjUxT/ZET5KPJoMCgLbQxEjlqjIih1WfQKR21kxTForZFQGWfQQqlCFBOGoA2qf//fyQfOn+31byglxis6h9pUfbNoHLGyCH1rnI+NZ2uNBlzbu/Mr//nkhUfeORr0x1sbP/vhQea5iXZz8+yWJCjFbcl5SRoBuFTVkRUY9AB1cku5j0nWJfGgghRQJ0ByhSHo/mi8d3pb65dfbkaZzsykVP9kTckYPr+E73ndxk+fffmhB++7dXN/euWptbN3K1ZGGJZwrfQtcW7VQrMjkMbTyVknLNL7pQln3UkDdAPLZ93v2Y21nhBbGJtGkGIC5tCxFXR6dSI5qHl1+xvO9QfD4UsvXhmeuUtxFJ2TGugEd2Uql8g6mV0WZvYjuZYS4g4TYGrJCUBCKd+5pp3PelYadrZsCXwCctKkp6LKqjsajwy5ejpVqkzel/Gz737rUoJXVuOE6ZVX0uqXOSb3kq/kcMpfuZLhpvyx9HRDqEPwigJTIPJKnl0148yL7Ph1NhxVvyqQ5o3cz+SYvHn+1y98+A3rRsuywFqDiKIueRWxnGIIAUQ8I2rdiVoKVYxEUaxOSqwsMTSm9YDitfWNy9emWutP3L/buerfk/Cl/85cH9++ubkxsprP3nb641u66xCsNFH0QXJM81bGI1o0brZoq9LsbK3t7G7tnDlTDdcErYytZQmSejORKHWav8wtAeZrh89vbG//09f+6+uPP/PfX70ghM/Gh5li/O2lZ370o5+1Pq8bulC+7c/eYo2xvera9Rtf/bcnHtw701nG7/zthXq+mM8W83k9ndXTeVu3PlFYSda6xOWa4ES5ZRV0ge26chLJEw33eIUkTBR4idnn9jbvuPOOrdOnIc9QZpZLWf6kGRBRDCFE780Pfnyp66ydW7cg67FUsLkXZlaRECvNl1bLs1TKiZRL45DVKHM+EyCHU+aY4nTpyuSN9/SKwVBok/IgfzNL5LaQO3vWKVOgNO3sMDN2BbIczj2tc5+dqiWbtczFUj6W0e1WlqsjkSnbqWVQk5yISdlF0y0IZLJ5hrlmxa8IyEhM0Xzkr94lkWSO3scYJTwhBB+imIcuwCGEPPnoQ/A+eCLF0Qdi9t6zVEWMIbE1BOkf+YUxpUieTKmirs73b956zWwhS+eykGSkvGprdVEKY2LkGNFoWVQXg+FSBEi8jeDrUsMA0r8S9xWloUQ1SUkTldO09pFZ5tBCYcWxSeGjAsiFDFpKj3K9u8Y37bXrt775je+XRgsAWelLPWlUxqDRqLNwyNBoovcCSMyW7ATIrgBFRHk7gpBPBCME8h5tAmoMOS/fGZPjj8m+Y1HIg2UJqKV3yOxR9BnlJSq22OshorbFuPVENKuP20rimyQ9y02uHjGbSjGiLMqzQ2KKS0YDeR+9l3kZnZFx8PKtjEcUmWOQwBh5iSAzNlWAJBe0LDozfTgENCZpDQGg7feryu4feUybAUs5zuld6Uuy6akWKNdwxivxYyYfKASxKlYCEH2rHKfxSMWYq0mCJ6ptZAdIQAOTtAGZZdtiYfOWlO71uG1kUaMgOR09GpaTgyarfF4sL634yWU/IHknXJLNHtnhkerykkSKQQIGKjR1qGcCRmxpUIlh4qyMZVBoyxwtCafwQbN3ikFyipqjByPbViw7VwI9gsjb7qkt5gCyQ0IgXCcg+QBMyFE+k3wwWJSASM5zDELuEMFaNFrV3s1naK3u91AjtU2aXCLm2lgQUIQYgUkQlJU4vLJHiwlF0qWVbKZflbYKsJCeGNuaWlf0yvH2Zh2CLHSSNgnJcoKzZqXlAitl2LVEEYwhJY5YWxuaJiwEVrmxIUXnmuidcEgpbQux7YuJ8I9YD8ZdB6cgI/iW0BSDstnfL9ZG3jtATKUmbo9YoTXB+b44oyoyuSD2NjJr4UjyYEm4RfREKqDDBKYADNTUHDwaa0cjEQ3fcnCgLRirrWzoxRjFydiSgjODsWiKb3O+AHSMpEH5+UyXZaJmIMkGS1yZUBtlCmqdUkobvTnu3dpvEMTud2RPR15EyUwUS1o5hlBPleyyWOz15LXepY0TAm3RGNkaEq+htDEkBcumWlNoyC2EMal+k5yKfWeKul+R8zJVZix6ojfesYkgFG9jHFhrT20PD25OZZGa+0i20IqlSpZbekZgEZmiNEVJIfh6kdShUEHct+kPUWMMQQXRNvIOTYHGiu7XE0GmpRHlzp0qRiljWWESFJcsjFgpUVuKyojgRtfawo431hGvYgIn/ZM4e5cEUlZJaa+k7BlbUPC+qRWiLnvS412L/cqUPUmNCJiREg5Bl31ALQUenMzV2LQhLdFDFAHqQt4TRZQ8ymIzIljxM1KPrPslsYj8xs7WwrUF2hwqEUqhB4TU0dMPG43oF1OZd9FDraNrBPLaGmqrfCMk01I0sa7txnZ6R1SxlWdRS09LRkF4qZG9D4uFHY2lhClikVpnDLKATJZBSI6GmjkAjkdr4wp0EKCdC0v77qyAqNvu/R2mDbdII6yjiwAAAABJRU5ErkJggg==",
"webots_deer.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAYAAABAOmveAAARsUlEQVR4XmN0Dan/z8DAyPD//z8GGPgPYvyH8P7//w8XZ2AEqvnPBOaDxBkZGVHlwDyg2H9GBpDcfwaQ3v9A05kYUNhAc5iBUmBZqBFg9ch2wUxmBLmOEcltUI1Au5Fsh8uzMDAxQ12OkEbWDnQaA9C3YKcx/mdm+M8I8Tjjf7BNDCC1EGczM4BMAnufESLKyMgE1guyAGwOAyPUy0APghwEMQLuGEZsLmRABYwQwxgwAFQzEwMTxHAQDWEzgUP0H5D8B3QQyPn/mSAhzgDk/wcGACMoEID6IDQzA4hmApkDFgd6C6ieAaqGgRHEB6phZgEyWRgYwPpBbCYw+/yBmQwQtUCnMjExwPXBzWKGiiHRzCDzUMUZwe5hYmABOQTZt6CkBMLMDECPgpIAIzTJQUMP5F1Q8mFkYEJKBkAmSBlSiIPUgWKWgRkqAQ1ZcMyC45uJ4fOddQwCPGwM3B8PMXwXtGeApTiQDrB+aHRA7IMmJKitsJiCxTkjKBCBBgA9BExGYIczgpMHIyMslfzHiNX/DKB4Y0bKDyC1EHWMWNILEzBZQvRArAUnTUQ4MAgJCTII8nEz/P75jUHgy2GGj3xATzFgpjtGBoRbYB6ApTy4e0ESwBTEwsjEwgDLsiBFIL9B+CAV0IICnlmZkMOHAaIeEgPI/kEuMFDzDihAwDYzfL21kkFSXIzh8bN3DLcfPGYQ5udlULRyRPMOKKkzMPxHTUIINyClCHDBBuSzsDAzQTX8h+v8B/QAEyOo5EMNLXBEMjBAY4gRzJL4vIuBnY2VAZx0gUJ/fv9heMrpBLUUZDa0EAGXSjDPrGWQk5Fm+PL5E4OQTgyDuQ7MxbDkCaERiRVkBiieYO6BysAcxAAtCUEeYmaBJjmomaDQZUIJEQQHElIQwz5dW8ogLsTN8Jedg+EvAwsDCysbw8+fPxhABYfMr0MMrCzAjA+1H5wc/0OS8t9//xh+ywoz/Pz1i+Hf39/AMoAJJVb+I8IfETPgEpUR6mxGaICC4wSimhHm4v8MwAhiAtcZ/5DqIZAqWCpDDhNYrr1zdCbDq9cvGXg4VIBCTGD85+8/hl+/fjCAAoSNlZXhH9ARoNiHFDrQlA8qCYGB/efPX4Y/f34xfP/xi4GXBVJtgJMpAxPcE0ygAGD8D3E8NEJg+fQ/JMIYYAXTf6guUDXBwgI28B/QKGZInQINIoiHkFMvJImBxD98/szAxinE8PTVewYhvp8MvDy8DGyszAx///5h+A10LMgToGTIAixeQeYzAZM1MzCvAkOO4R8whkCxBPKUoH4CA3oF/R9cKf9ngFTojAzoiQUsgpKEkLMUIwPQQ6DKjxE144GjiIHhH6jIhvkJRIOKxn//GWwCKhn+//vLcHhDFwM7CyTjcnFyAusARoaPX74xsDIzAmOLBZyvQMUpyGPMwHqIGehBNhZWht+/fzN8/vaTQRCc3BkY4AUKKEb+w/jA2AJnP0hAwlINWC2sKoG1SIBKIE4FJjkWNmDB+p8dWMv/ASYTBqQC4j8w1pgYEMYB6zcg78/jZQzMf5kYmEEeAVZ+D168Z1CVZWFgBnr2FzBJXb/3hEFKVIABUi38Z2BnZ2VgBSZBkKdAqV5EgB/o2Z8MIqaJwFgA1Xb/UOxggAYmSBCSDEFuYkRRg5IN4BEE8hUDA8uP+3sYeKQsGBi5BMEeQoliYGyA8iPIW3wM7xk+PtrP8OnLdwZ1FUVgEmNkcLdmZNh1+CoDKM98//mL4e6zjyA3Mzx/8wFsDCNSZgVFMCg5WelzMXz/+ZtB4vcbYArgZvjHzssAa8cxAjPHP1DpCkyWTEyw+pEBtdwGl5bQXPP/P3J6A5c6LHysfxgY3xwGhiQXA6OQLsNfTnGGX8D4Yv37k+EvCxu47cb64xWDAMsDBnYRXgZVJVGG79+/MzACSzVpSREGTVUFBqZ/fxiuPXjFkBhmz/Dj23eGD+/fM7BycAE9CmrKsDL8BBblW/eeB7sM5AZdbXmGX58OAz3CwvAXKPCNQZXh65cPwGTMyMDJxcrAKWcCDFxg7DEi5WFwUvkPLpfAlSkDcqMVEWcsXJws4JY2I9Nfhr/vzzMwfwS2h4BpHFRKgdL8z59fGdiBjr8H9ISUGB84Sf74zcDACWykvvv4EVhgMzB8BcaOta4kw/UbVxn+MrIC22z/GeQFRBi+fP0EzKTfgEnsN7i99/fPT3C+ApnNxSvA8P/vfwZeXn6Gp0/vMggKA/PWXy6GDx//MXy+fxQYSEwMgurWQPtYGSCtAWCrHeg/ZqAn/wBj/g8w0Fn+/QYHCqgDwAiM1f/AopGFCWgoEysjw29gvfDnDzA9gxp5wKj/DVLw6w/Dn79/Gf5xczNoyIgyfPv6meHbt18MHMBA+P37H8Odex8Y2IAFgLgUMI3//wv0ODvD5y8fGZiZeRlevHzNwMT6jeHPly8MbGzcDCa63AzcXMAaFFhc/wUWKIwMHAxc3CwMX3/8ZJCSFgdWst8ZFOWlGF69e8bw9Ssjw9vX3xm+3z7C8BGYPLlF5Bm+vnsOLKWAIfn/DwMnBys4mf8AupmFhYOBVUiKgZVPAlgHsTOwXHvPA8820OIeXPbLcrwAOgxY0gBTzbcPHxhuf34HTEa8QE2/gSH+AVj8cgE9942BW4if4fXbj8Cim4uBFej537/+MXz7/olBUOAPw6/fPxl+ADEwzBh+AYtzXmBA/QcGNxsw3wgKcAJji4Phy/c3DN9/sTBcfQbCLxgYYI3e/2wMQIvARd63l48glS8jqM7kYPjxE5jEfgCLQEZgSfoHqOX5CwZGIAZFJaO7bwxyZQNOmeBc+P8/ZlEO9To7Ozu0bGVkYPt8i+EXnwYk44Ka/CB9oJrvPyR4/gPzIqx1LsryClgaMjE8/ynIAKkkQRUKrGP5H953YoBlEmCsMzBCKn4GaAEDzi3woheYDMH1Gyw/AYtthv9/IM6ERQ+0IfIfxZtQDlTwH0gLtPL49v0HAxPndwYGtHgGV4xg9RAHg1oiLOzMwCYRsPT6/gvoAlDT6B8DVgAPEFAP+R8DuLJlYgUr/c8IykuwSh4o/w9iBtiF/8EeAqkDF6rAEAJVsP+QiklEImSAKQMV96AmDMQEoANZgW05oKUgg8Eh+w/SnQcazgjq3DEyQPkMwPqIBRJcwJAHdfkRaeA/xE5GiENBtQ5YDmwGE7h9CK9TELUwA0gTJCGAdEBKOmAX8h8DYtwA2khiQIp+ZJ8wQCz+9w8SqyBjPn8H1visTxl+cEgBzYd6Bhrs4HEKcMOLCWz563cfGT5++szAwi/HAOknweIH1iMGOQrSrgG3XRhhrWhYGgOHDsN/cF2EcDe4lQD1KEu0OhcDOwc7sCRiYwDlDVZg/mADYhZWUJJgAre9/gFLpd8/fzKAugZ//vxm+PsHWPoBi3ZQE6b8/EuGn/8YGbJt2cGFCQu4lc0INg/kXFDRD6JBTaCsGScYJqf4M3RsPc5Q6GIEzar/IS1uJmheYUB4AtQmBGFwjP2HNFRhcfgfOakxIABjY7DrfzY2ZgYOYJ+Gg52NgZMTWJxycTHw8PIycABpLiDNyc0LLOHYgc39f0BP/YR4CNT8Bxr6D1iy/fn9Cyz2F8z+A2X/ATdWQYEAUgNWByoFwYHxB9jV+MXw8fN3YFHOwSAjK8UgJSvLwM7FDWxSgcZtoO1AZlDlyghJcuAUxQhNmgyIAgTqF3DjCKT03JKJwHEPoEbQwAOoJGFE9i40bYIUgqMUJvkfw0BwFvsPEQc5/i8wJn98/cLw4+tXhq/AjtwXYAv906cvDO8/AusyYM38B1iW/4eaCU5oQLtBnUpGpDzyH9ZTAudXiN2Q3M7IAC3HoQ1baJIFphHGIm+H/8xAQ1hZmBjYgBgUU6zAVjAHMBmyApMdOxsbMPmwMoAal6zAWAQlIVBDFNTgZAJ1D4CtCB4BQQZQEvjw9jXDm5evGF6/fMnw5t1nYKsa2Pr4B8ot4HTEAGrbgUZnQO00kJ2MjLAAgzoW6Km/QJf+BbsWOCDFwMgAy86wHgOoTgMaCR33A4c01NsgM4ClHCvYUKA+YCvhF1D17x9/ILYzfAVHFSMwx3EAkyQ3Fwew8uRk4OKCYJCHOIEtCG5+QQYWNnZwiHHy8TFIKqqA9YMKBFhSA/WT/gErVlBfCJQfQXnwPzi5/mMAJVNwUgUmx79ANaD+FCSZ/mcA8f8CO46g2AT3o0BJGuhGUIsGJvYP6Dtw/wpcgoO6D/9h0ccALu24gM0KEWEhBgEhPgYRcUkGfiCbR1AYKPmXAVpXAttgf8ElGiRBQ3uZ4CKWAVrk/weHICOwgAD1g4ANFXAShdVt4CEnBkZwjIHGAiHpH5psQON2jNB8A09y4DTHgKjb/oMrdtB4ILyIZ4D4g8XKQo+BF5hkuPn5GbgEhIHmM4KTD7gUgdb6/4AlHNhiYM8T3mliRJRK8CYww394VwBoCMTBjFBPMv5HShoMkJYGyIf/oLURIzQsgCUqJBczQlsjUB4sf8NCBZxd/zLAACOUwaJiYAzWCPE9A7gkA/scmDfAo5FglYzQ1ggk04N6q/+B3W1QsoIZ9B+YNMCjr4wQtQz/QTHHCCk8wIqglTQsIP5DQhlkMFiGERSQ/yGBCYptUDHOADcdGGZ/GRgYEX1vRnhAQeo4mD/BQzOQpMQMbbtBWl6MoNgBO/o/NC9BTGCExCywiwAa3GCBhCIjLHgZ4aUPA7TEY4ClQgZI/QJvHUDlEckIGBBMkFgBV5Tg1AGxG2QWhPUXkpTBgy3QFg0jImBAhQwLEzsnNK4Zwfni/9+/4LQKLh7/Qmpj8MARLICBYwOg4hXcx4Y7EsgAjUvDHImUEEAJDR7OsDFvkFnQpAXqpTLCY4IBlr4gMQtyCwMkSMCBDiKYUJP6P3AF+x/iB6A8CxOwogPXB7BQBqdVaF5ghEQnIwMk4BnBzX8GSAb/A/E4OC2CWtmMEB+Dk+n///AChBFWioLsBDkIWD+B3Qi1D1LjQ2P2HzgYIXXL//+QYh6UFEHJ9N8/eIEDDy+g82CjSRB/A0u5f9DhJdCoP9hwkEZgWw3sCZA9IIMZINH6H1oCgZPJP2gdApSDjJpCkgsof/0DBRJIC7CUA2kFjcQy/oc6CGQ+KDZBHgLSTMBRIHDeY2CEtylBxT0DqIMJikVQFwIYUwygAgkpJsF1Gjjq/0KDG5IoWUD1Aii/MDD+QUQ3MyvDf3iOYgEPWUEGYpmgmiE9W2irCxx9EE/+A4cqEyhmmWDZC2QrSB8zRACSdhjAngDHGrRBC9IDSgHAwGSCioOjBDpNAy4EYIUC1AxQHcTwH9rSZ4B56A+05QyNWgZwKQVqdUNKHIZ/kD4JOMagIQtKVrBQBYc+qE5hhngZnGeALQh44xGabMHFOMhOJojHQHkVJAZJX6BkDMnwDCyQfg84/kFugiXV/xB3gM2FjReCCz3I/BS4ZQ+MSRZQpgIXudBkBayeIUmcmRlcTMImkkAxAC6ugX0cRmgyAOc9ULoHJdH//8ADIYwgz4AyLjAp/2eC5EGG/5BKFNw5AYUuqPsBopHrMpCtwMCDjyFCMwrjP0j9BTaLEZxpQHULA3h4jZERmgyYwPkaFOjA+SEgxcwCycSgUGBmg5S4IEuBGOTZ///+Q0odUIJjYYUWndCZOZApIM8zQMaowWPkf0B1BrTSg5VMDFDPgd0AZYPMB6UAUKqAll7gsTwmaLkHjtH/KGUguMQF2QeSA+UxkBmg/McIqfdYQO0qWJ3BCB2wZ2QGDS0yQtI50KGQeWKYp/6DMy+obQZPjoyQgpkRliSBfFDDlQHWjAGH/h9IvgRZDIsGkDgjJLP9BxcKwHAD1n1gc8EeByd0BrC54NwLKX1BeQsqA56uBLfS/0OyCQu4YfkPkjEh6fo/tAT8D26z/f/9GxKdrKzgOVLQBBnIMFDyA9dF4GQDmqcFRTtwyAtUwkH1MECTBCPU0eCWNjMkD0E8zwCdVwLlAxawk0Fm/gfFNrRgANeLwGkacAHAAEthTAyQVgwzA8wcSF5kZAAA91gzEqUBoxAAAAAASUVORK5CYII=",
"webots_horse.png": "...",
"webots_truck.png": "iVBORw0KGgoAAAANSUhEUgAAADQAAAAnCAYAAABAOmveAAAPmUlEQVR4XmP0iev+z8DIyMDAAMT/gSSYzYAB/jMygJUxMTAB5f5DOIxMGOoYofoZGSFmgfhg04ECCDlGsD6QGiaQGVCzQaJMTEwM/yGyDGB9TFC1IDEks0HuBXP/w+xhALuJhYmRGWjAfwa4AiSNIOZ/iHlgb4B9BLacGRIAUO+AlTDC/MkIDhtGoHNgnmKAeo4JFnAM0PAAexJsIQMTwnUMsACAOIURzv/PAAp7Rrjd4ABiRJgFUs/CCAwRmBKQt/5D3AMxBEjCYoYBBhjRDGRAjVWYYxiRYgSklQkY0oxIsc+I5BlYTDBCAxPuIbBbwKIMkJhmBMceIxPCk6gJipGBBWQRLBZgToUnDVhYQHVBjYYECdSDYLX/YSGHlMwgrmOAxxIDImlAnAMWYIA5CGQOKLlB4gvJE0Dmf6RAY0Qz5z8SH6SXhZkF5GtGBmQAshAcvQxo4rDYQQs5aMAyMDCg5huYdkZgPsEIdbAZ0NiFeh6c7FDMRpYHZwqEMxmhdiHZCfYQKGSYQQ6FRtN/uB/AOqBuAgn+hzuYgRHNcAYGpJCGpmmgoqfX98EdICklAWaziegyMCIFFNizUA8xABMULJYgloGDCKIankQRSRy5AIJZxMLCAiwU/kMKBZAzmVATJdQTEIMZoBmdAWEbwiNAsac39jOgAEQ4wIV/vbkMZv+HivBIGDFAAggkwgQODZiHwSKMkKjE5niYOljhADKShZmJCcUCmNPRUhsw8JigFjPAowM5BhhwASye+o+k9suLcwx8UkYM6DHFCE3ekPCFuRGRSphAgQsq0v//h7sHpJaFiZmJASVSoMXcf2htwPAf4iK4GkZGlCTDQAyAJuP7W1djVS3GtQ0qDlEoG18P5sPs/A+zEVwi/4PwGKHu/v8fpWBhYWaG2gYNSVjU/ocGIyyk4C6BVqaMWJImPr/BPHOHjReuTOXXZzD72B8mBi5uPkhyA7ljYSM81BWSGhABCE59TAyI5AdSzAz2ECzTsLAyQ6OTCAeieIKRgTQADzBMbdDUxQCtZRgucQmCHa377R0DCzOsNQJx8j+m/wxMwFQDCleQzL9/DAygVPaf4R84MQGLbagGjFIO1WJGeETCGKT6CGYeHn2Q/I+wGOgHFhaINyFZhZEB5L9/DMCC7M9vhncvHzK8enYPKMbMwMfLz/D62S0GYCEHsgBmErSkARr5H6myxAhTsBbyPAQJGORi4T+0RQVLNAxI5RFQjpmZAdQ8Y/j7i+H//99ARz9neHH7JMP3798Y/v/7z/D911+GP38YGf6AffwP6CEWaAyB3QciEA1OEO8f1DfwGPoPEgHKMDExkAyA2pR+fIJqQ3gKFjQQGioOpS7uXcHw8+dPhg8fvjL8Z2Fl+MMKLJT+/mb4/wcSEH//AJMcE6htCSzGgJ5ieXZ1L4OYvC4Dq6AEAyswYf7//wfcUPwLjFZo3IEdgGg5MDEwkBc5oHjHGgbmDL8ZGL69h8jBAxbCffvhA8P3339BVS4DAzCZASOJgQXoqb/AgGX+DywQgHnqH9DNoDYpDycTA8uvLx8Z3tw5BcxQfxk4BOQZxGUUgCHBzsDGzsHAyMQKL1H+gVMExCfg2PpPavzg1nDs1y8GDk4ueEAZ/f3LAON8+/UXHBD/GYFBDIwBfl4OIP8vAyMLB8M/YCx9+vKNgYODnUFcgJ3h64+fDCzsQN/++P2b4Q/QkLcfbzPcvHaVQYiHE+iZ7wxcXLzA9Pmf4S8wjTKyszMwA9Pzf2CxwsHFzfDzxw9SfcRw5ecPcO1maKDP8OfaTST9/yGxBy2YzjEzg+VAwafMzw7MK7/BdoNKtF9/vjNwsHEw/P4D9CDjfwZRCRmw2u+gXMDJwMDy6RcsMzIDo/I/g4AA0OdgJZwMn3/8gSQ3kBIoG9Te+vYdkg/OnDgGlldVVWK4ffsexFFA0sTCDcWzn94/Zrj1ExEA5y9cZFBWVGDgevocLVBg+YcRHEEgHgsrKzAQ2MBmAxMYAyc7JzA1sTOwssHUgtzICGsGMLD8//8PGhqMcAeBBMClHANMIczTkIwHqo+QE9Dr129Q9N6+foLhJzipMOAEr4B6/jCzgMKHgZOLiwHqCGj2RGRScIsF7EaYB4A0WB/MTVAroNIsDNC2EJj/nwHaewUxQFUVNvcAPQluRiC89AGYcZFLio8fPzEQAl+/fmUwN7OAazt+/BhcCyhpm5paQPiIJgtUHlgo/fuLYjyy11jAzv4PCWBkif8oWpCLtf+oQQLk/fv3h8HK0obhxMmTYDkQf2FRCkPihAUgWYZ5+YkMSRMXwpMFAzi8/jKkmikxgFLCX2D+RfYQKNXE6kqCY4sVmOTYOWH59z8wT/9mmHbyEc6ClnFDY/F/WOz8AxZl/4C8v3//MfyDRBIDqH8C7k0yMzOgt9/+w0IPSMPUYcQMqEgEyv8D5Who0gE5GKwXyAfREDYDOLZAdoCKYFAsMaPZCfb8nz8MIDcyQO0G2csEbR6B9LKoGxhA5ICR8A8YUt+BxeCbl68Ynj17zfDs9WeGL78YoK1bZKcyoiYHmBQjqjikvwLJd+C8CnIEyFP/gfUKMNkwAmllWUEGXSNdBn5hEQZGpMoanGjAgQDRzwBLPrA6A+oh8JgIEzSwgXKM6xtK/n/+9JXhw8fvDD9//wNXYP9BmsDdZkYGpLY53EeMDDgAIyNqUoB6kJEBGt0gbeBYAeYBEA10MMyjQoLcDNKykgyCIsIMzMBk9u/ff3ig/f8HCoB/DPAUAXYWI5bEABwkUTc0BJc04A4WqPwHq/sPEQM1KcAxBzTsLyQjMsJGb4CO/QesC/4C6zAmUNIAjacBHQgOZUZIQIDYTCygkgyY9n/+giQpoDrw2BvQQ+DSEuRYoNn/oYXTP2By//r5C8O9Ww8Z7j58w/DtJ9Tz8NgFxx3Uc8g5HVJKM17fsuo/pPnOCE7EkECFlGT///5D0siIiCxQ0gC138HJB1js/4c0jEAKIAUgqFEJHJIEFa+MELP+AT3OAHI8IzQJg2wDByBkSAZSFUBLUJB5wBj6DWzDvQTWVY8fv2Z4/gaYir79YgDF3P//IE9CPAZPimDX/2cAVwT/oE0NJmZgJQYNeZBnmFhZgAOMQAcjt1CBDgIlCXBEAsX/gvwDihmg40Gt33+/f4LzHCM4xv6CCxUmUBoHjl0A2yvA5sofYF4BpXlgC+Q3sN3Ixs4AycT/wK0BUIEE6S3/BVb0LAwyivJALMfw+9dPhsf3nzJcu/2C4dmHn5AaEhRADAwQP4EDFdibvbZhGbjZCnLMf6DHQDHExMaGyDsghSBxoKOYgGkbUqIAQwdo2D+gg/4Akx24EgZ66N8vUAj+Y2Dl5GQAJTVQMwk0CAPpFUOGy0ClHcjRDED1jMBOGSjZggMHlBRBgQUssSAlLaQUAJnxH9ik/v8P0r77B6S/fvjIcP/+c4bHL94zvP34kwHR1QHG0N/vPyBpG5oPQDkN5FBG0GgQKG0DHczIygzuWDECW7X//kDyx19gnvgLTBKggGAE5pNfwP4JsNHFwM7FAczULOBeJChGQCUZIyy/MIBdzPAPnNdANjEB1QK9BY7lP0CHAfs1f34xsLKAkiszw29ggIHNh+VtUPUA7AXwCAgyaGtzMWhq/GX4/Pkrw4P7jxkeP/3A8OMXsD/ECNTMCGp0Ah0PyuDgYhCYEP/9+A32KDjZAWMHFFJ/f/wCRy8oif79A0nHzGyswL7XT3CXgxnocXBAgJIBOI8BjQAnBViZ+48BNBQPirW/kHIZEroMoG40sNEC6ucA2aBWEzMwnzADUwqwd8Dw9zeoQAGOJYAKJKDbQJUkE9DdoPzED0w1uvy8DJrafxjevwV22f8BK6r/QMexcgGTCTBq//z6zfAPmPmY2VkZQNaDPMEIbLGDxBmgye/Prz/gAoKZA5T+gaECrMn/AWMLVEIxg4p7JuhABjBC/v8DhvY/YOAAWxP/QXkUXNj+Bxfv4K4iOL2BSwoGcPICugeUXP8xA7svfyCtbCZgS/8fkP0H5FZgSDABO3qgkpUJGAD/mIFuANrJAoxNfqCBLMysoObcP4bfwLYVyFGg4hsY5wy/gbHBBmw0gppGf4H9DAZwyQbMyMDkAoohVm5OcHYExS4oif4GepKdhwuc1MC5BZTmQfUSE4gH7EYDS63/4BENUFsMElP/gI77B07BkN4vKOmCSjtQkmT6+Z3hPys7w1+gPlBSZ2RhY2AFpiRQafkP6BFQCmHiAEYCMDmDIgVUNPAI8zOw/Ab2a379BIYE0DJwTxBoAAsTOwMLDwfDH2DJAurqggoCEP755TsDMzsbsPRhYvj9DWghMGQ4gdH9F2gBCwcbAxMrpDCBjZaD0j8o84HiA1Q3/GNiAccPMBswMAJjmxFc/AKFQK0GUICBUgDQg6AYAOczoLmMzND2869vDMAMx8AI9CQLyx+Gv6wMDP+ASfE/cDoIpIYRaAbITSxvX7xmYGGE1Bt/gC0FVm4uBkZgEvzz5jU4k4JLNlB6BUY3uIH39QvDH6BnQGEMUvvx62eGP8ACAlSUgzAoJkFDfYyg9iC4yASnOwZGUB4FegjUAGD89wdsFCg2/gBj6T+0YgW3FUHKQfUfyJ/AQGaCFs0MoEoe3MoASrCAfPMP2PH8A64GQHkJ5E5QTLFs3XkSmqoh5TmoYoT0hRgYkOthUCjDhkVAMfD/P5I8KNnA8j0jZGQVNErLCA31v9DRV3BMAdUxAQMQ3JgF5yRIrQMyDzS1gxiahuQrUBYDzfL9A7uGCdq9YYTENCO0UoYkfnDjmeUvaG7uP6xbAvECbFoD5gFwGxGkCZzeGcHaweb/h6gHUyDDgRjkEbBnQCH9H5L5QTEG8hQz0CPM4KYTEwOk54U0aQPVywBMhojBZkgH8y80ZiA9NFDtBfMPpO0IdRHYSSygSgskAG6SM0IMAGdMUN0JayCCG6r/weke3NuHBB44qTJBZwDBDgQFDLipA/EzdHqUAVREg/LTH5CZQEFmqOMhkQqx8z8swBigKYUBqakFKsWYGKEVKAM09CF8yEg20G1gy4CNU5b/f+EhDmvNgpI+SOF/mOH/GKCDgdCohiYVcISD8gAjVPw/LOQg6pmgHgSZxcoETZPQmAMnMUZIuw6qjQE0lsvICNH7DxLtwLwGjBFQSfYP4kGQu8HzWYywVMUIbTn9hyS5/+Ba/D9YFjbTBvLYH4gQuM0EC2l4rgHKgdSA6hGYwyDDj4wM4ClOsAsZGf7C/A9VzwBNlrCmCiMTxDH//0NiCTSvAEqa4M4lOIb+M4ACgwnhAAZwcgYKgtvFDAyIvho4vBgZAJlM8MCIVE/uAAAAAElFTkSuQmCC"
}
print(f"✔ {len(SAVED_IMAGES)} saved images available: {', '.join(SAVED_IMAGES)}")

In [ ]:
#@title 6.2 Choose and classify the images
IMAGE_SOURCE = "upload"  #@param ["upload", "my saved images"]
SMART_CROP = True  #@param {type:"boolean"}
SAMPLES = [("car.jpg", "My photo (car)"), ("cat.jpg", "My photo (cat)"), ("bird.jpg", "My photo (bird)"),
           ("webots_dog.png", "Webots e-puck camera (dog)"), ("webots_deer.png", "Webots e-puck camera (deer)"),
           ("webots_horse.png", "Webots e-puck camera (horse)"), ("webots_truck.png", "Webots e-puck camera (truck)")]
IMAGE_TYPES = (".png", ".jpg", ".jpeg", ".bmp", ".webp")

def choose_images(folder="my_images"):
    """Ask the user to upload images. Returns [] if cancelled or not in Colab."""
    try:
        from google.colab import files
    except ImportError:
        return []
    os.makedirs(folder, exist_ok=True)
    info("Click 'Choose Files' to pick your images, or 'Cancel upload' to use the saved images")
    try:
        uploaded = files.upload()
    except Exception:                       # upload cancelled
        uploaded = {}
    items = []
    for name, data in uploaded.items():
        if not name.lower().endswith(IMAGE_TYPES):
            info(f"{name}: not an image, skipped"); continue
        path = os.path.join(folder, name)
        with open(path, "wb") as f:
            f.write(data)
        items.append((path, f"Image {len(items) + 1}"))
    return items

def sample_images(folder="saved_images"):
    """Write the saved images (embedded in this notebook, cell 6.1) to files."""
    import base64
    os.makedirs(folder, exist_ok=True)
    items = []
    for name, label in SAMPLES:
        path = os.path.join(folder, name)
        with open(path, "wb") as f:
            f.write(base64.b64decode(SAVED_IMAGES[name]))
        items.append((path, label))
    return items

def find_object(rgb, margin=0.12):
    """Square (left, top, side) around the object in a Webots arena image, or None.

    The arena has a pink/brown floor, so a Webots image is recognised by its floor rows.
    Pixels that differ from the background colour of their own row (sky, far hills) and lie
    above the floor edge are the object; the largest blob gives the bounding box.
    """
    from scipy import ndimage
    img = rgb.astype(np.float32); h, w, _ = img.shape
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    floor_rows = np.where(((r > b + 20) & (r > g + 5)).mean(1)[h // 4:] > 0.8)[0]
    if len(floor_rows) == 0:
        return None                                              # not a Webots arena image
    floor_top = h // 4 + floor_rows[0]
    diff = np.abs(img - np.median(img, axis=1, keepdims=True)).sum(2)
    mask = diff > max(45.0, np.percentile(diff[:floor_top], 70))
    mask[min(h, floor_top + max(1, h // 25)):] = False           # below the floor edge (keeps the feet)
    mask &= mask.sum(1, keepdims=True) >= max(2, 0.05 * w)       # drop thin horizon lines
    mask &= mask.sum(0, keepdims=True) >= max(2, 0.05 * h)
    lab, n = ndimage.label(ndimage.binary_closing(mask, np.ones((3, 3)), iterations=2))
    if n == 0:
        return None
    big = lab == 1 + int(np.argmax(ndimage.sum(np.ones_like(lab), lab, range(1, n + 1))))
    ys, xs = np.nonzero(big)
    if len(xs) < 0.02 * h * w:
        return None
    side = max(xs.max() - xs.min(), ys.max() - ys.min()) * (1 + 2 * margin)
    side = int(min(max(side, 0.5 * min(h, w)), min(h, w)))
    cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
    return int(np.clip(cx - side / 2, 0, w - side)), int(np.clip(cy - side / 2, 0, h - side)), side

def to_input(rgb):
    return preprocess(torch.from_numpy(np.ascontiguousarray(rgb)))

def classify(batch):
    model.eval()
    with torch.inference_mode():
        return F.softmax(model(batch.to(DEVICE)), 1).mean(0).cpu().numpy()

def predict(path):
    """Return (RGB image, 32x32 model input, probabilities with the plain centre crop,
    probabilities with object crop + test-time augmentation, crop method)."""
    rgb = np.array(Image.open(path).convert("RGB"))
    plain = to_input(rgb)
    prob_plain = classify(plain)
    box = find_object(rgb) if SMART_CROP else None
    views = []
    if box is not None:                                          # object crop at two zoom levels
        for m in (0.12, 0.25):
            l, t, sd = find_object(rgb, margin=m)
            views.append(to_input(rgb[t:t + sd, l:l + sd]))
        method = "object crop + TTA"
    else:
        views.append(plain)
        method = "centre crop + TTA"
    views = torch.cat(views)
    if SMART_CROP:
        views = torch.cat([views, views.flip(3)])               # test-time augmentation: mirror image
    prob = classify(views)
    x = views[:1]
    small = (x[0] * torch.tensor(CIFAR_STD).view(3, 1, 1) + torch.tensor(CIFAR_MEAN).view(3, 1, 1)).clamp(0, 1)
    return rgb, small.permute(1, 2, 0).numpy(), prob_plain, prob, method

header("Step 6.2  Choosing the images")
items = choose_images() if IMAGE_SOURCE == "upload" else []
if items:
    info(f"{len(items)} image(s) uploaded")
else:
    info("Using the saved images: my car, cat and bird photos and the Webots e-puck camera images (dog, deer, horse, truck)")
    items = sample_images()

header(f"Step 6.3  Classifying {len(items)} image(s)")

fig, axes = plt.subplots(len(items), 3, figsize=(10, 3.1 * len(items)),
                         gridspec_kw={"width_ratios": [1.4, 1, 1.6]}, squeeze=False)
IMAGE_RESULTS = []
for row, (path, label) in zip(axes, items):
    rgb, small, prob_plain, prob, method = predict(path)
    top = prob.argsort()[::-1][:3]
    p0 = int(prob_plain.argmax())
    info(f"{os.path.basename(path)} ({rgb.shape[1]} × {rgb.shape[0]} px) → {CLASSES[top[0]]} ({prob[top[0]]*100:.1f}%)"
         f"   [centre crop only: {CLASSES[p0]} ({prob_plain[p0]*100:.1f}%)]")
    row[0].imshow(rgb); row[0].set_title(f"{label}\n{os.path.basename(path)}", fontsize=9); row[0].axis("off")
    row[1].imshow(small, interpolation="nearest"); row[1].set_title(f"what the CNN sees (32 × 32)\n{method}", fontsize=9); row[1].axis("off")
    row[2].barh([CLASSES[t] for t in top][::-1], prob[top][::-1] * 100, color=[BLUE, BLUE, NAVY])
    row[2].set_xlim(0, 100); row[2].set_xlabel("probability (%)")
    row[2].set_title(f"Prediction: {CLASSES[top[0]]} ({prob[top[0]]*100:.1f}%)", fontsize=10, fontweight="bold")
    IMAGE_RESULTS.append({"Image": os.path.basename(path), "Source": label,
                          "Centre crop only": f"{CLASSES[p0]} ({prob_plain[p0]*100:.1f}%)",
                          "Prediction": CLASSES[top[0]],
                          "Confidence (%)": round(float(prob[top[0]]) * 100, 1),
                          "2nd choice": f"{CLASSES[top[1]]} ({prob[top[1]]*100:.1f}%)"})
plt.tight_layout(); plt.savefig(f"{OUT}/image_predictions.png", bbox_inches="tight"); plt.show()
show_table(pd.DataFrame(IMAGE_RESULTS), "Predictions on the chosen images", fmt={"Confidence (%)": "{:.1f}"})
print(f"✔ Saved {OUT}/image_predictions.png")

**Notes on the Webots images.** The e-puck camera gives only 52 × 39 pixels, the animals in Webots are 3D models (not photos like CIFAR-10), and the camera sits about 3 cm above the floor, so the robot sees the animal from a low side view. Because of this *domain gap* the CNN is less confident on the Webots images than on real photos, and it can confuse similar four-legged animals (dog, deer and horse); the truck is a simple model built from boxes. Fine-tuning on a few hundred labelled Webots images would close this gap.

---
## 7. Optimise for the robot

The e-puck updates its motors every **64 ms**. If the CNN takes too long, the robot moves late. Four versions of the model are compared on the **CPU with one thread**, like a small robot computer:

| Version | What changes |
|---|---|
| **FP32** | The trained model as it is (32-bit numbers) |
| **Dynamic int8** | Dense-layer weights stored as 8-bit integers |
| **Static int8** | Weights *and* activations in 8-bit, calibrated on 2,000 images |
| **Pruned 50 %** | The smallest half of the weights set to zero, then fine-tuned for 3 epochs |

In [ ]:
from torch.ao.quantization import quantize_dynamic, get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx
import torch.nn.utils.prune as prune

engines = torch.backends.quantized.supported_engines
torch.backends.quantized.engine = "x86" if "x86" in engines else ("fbgemm" if "fbgemm" in engines else "qnnpack")
cpu = torch.device("cpu")
fp32 = copy.deepcopy(model).to(cpu).to(memory_format=torch.contiguous_format).eval()

def size_mb(net):
    buf = io.BytesIO(); torch.save(net.state_dict(), buf); return buf.tell() / 1e6

def latency_ms(net, runs=300):
    torch.set_num_threads(1); x = torch.randn(1, 3, 32, 32); net.eval()
    with torch.inference_mode():
        for _ in range(20): net(x)
        t = time.perf_counter()
        for _ in range(runs): net(x)
    torch.set_num_threads(os.cpu_count())
    return (time.perf_counter() - t) / runs * 1000

header(f"Step 7.1  Building the optimised versions (quantisation engine: {torch.backends.quantized.engine})")
dyn = quantize_dynamic(copy.deepcopy(fp32), {nn.Linear}, dtype=torch.qint8)
info("Dynamic int8: done")

calib = GPULoader(X_train, y_train, 100, shuffle=True)
prepared = prepare_fx(copy.deepcopy(fp32), get_default_qconfig_mapping(torch.backends.quantized.engine),
                      example_inputs=(torch.randn(1, 3, 32, 32),))
with torch.inference_mode():
    for k, (xb, _) in enumerate(calib):
        prepared(xb.cpu().contiguous())
        if k == 19: break                                  # 2,000 calibration images
static = convert_fx(prepared)
info("Static int8: calibrated on 2,000 training images, done")

pruned = copy.deepcopy(model).to(DEVICE).to(memory_format=torch.contiguous_format)   # pruning needs standard layout
targets = [(m, "weight") for m in pruned.modules() if isinstance(m, (nn.Conv2d, nn.Linear))]
prune.global_unstructured(targets, pruning_method=prune.L1Unstructured, amount=0.5)
ft_opt = torch.optim.SGD(pruned.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
zeros = sum((m.weight == 0).sum().item() for m, _ in targets) / sum(m.weight.numel() for m, _ in targets)
info(f"Pruning: {zeros*100:.0f}% of the weights set to zero, fine-tuning for 3 epochs...")
for e in range(3):
    bar = tqdm(total=len(train_loader), unit="step",
               bar_format=f"   fine-tune {e+1}/3  {{n_fmt}}/{{total_fmt}} {{bar}} {{elapsed}} - {{desc}}")
    train_epoch(pruned, ft_opt, bar=bar); bar.close()
for m, n in targets: prune.remove(m, n)
_, PRUNED_ACC = evaluate(pruned)                         # float model: accuracy measured on the GPU (fast)
pruned = pruned.to(cpu).to(memory_format=torch.contiguous_format).eval()
info("Pruned 50 %: done")

header("Step 7.2  Measuring accuracy, size and speed (time per image on the CPU with one thread)")
info("Float models are scored on the GPU; the int8 models can only run on the CPU")
OPT_RESULTS = []
for name, net in [("FP32", fp32), ("Dynamic int8", dyn), ("Static int8", static), ("Pruned 50 %", pruned)]:
    if name == "FP32":
        acc = TEST_ACC                                      # already measured in step 5
    elif name == "Pruned 50 %":
        acc = PRUNED_ACC
    else:
        _, acc = evaluate(net, device=cpu)
    ms = latency_ms(net)
    info(f"{name:<13} accuracy {acc*100:5.2f}%   size {size_mb(net):5.2f} MB   {ms:5.2f} ms per image")
    OPT_RESULTS.append({"Model": name, "Accuracy (%)": round(acc * 100, 2), "Size (MB)": round(size_mb(net), 2),
                        "Time per image (ms)": round(ms, 2), "Images per second": round(1000 / ms),
                        "Share of a 64 ms step (%)": round(ms / 64 * 100, 1)})
opt_df = pd.DataFrame(OPT_RESULTS)
base = opt_df.iloc[0]
opt_df["Speed-up"] = (base["Time per image (ms)"] / opt_df["Time per image (ms)"]).round(2).astype(str) + "×"
opt_df["Smaller by"] = (base["Size (MB)"] / opt_df["Size (MB)"]).round(1).astype(str) + "×"
show_table(opt_df, "Optimised versions on the CPU (one thread); green = best",
           fmt={"Accuracy (%)": "{:.2f}", "Size (MB)": "{:.2f}", "Time per image (ms)": "{:.2f}",
                "Images per second": "{:,.0f}", "Share of a 64 ms step (%)": "{:.1f}"},
           highlight=["Time per image (ms)", "Size (MB)"])
print("✔ Optimisation comparison finished")

In [ ]:
header("Step 7.3  Comparing the four versions")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
names = opt_df["Model"].tolist(); colours = [NAVY, BLUE, GREEN, AMBER]
for a, col, title in zip(ax, ["Accuracy (%)", "Time per image (ms)", "Size (MB)"],
                         ["Accuracy (higher is better)", "Time per image (lower is better)", "Model size (lower is better)"]):
    a.bar(names, opt_df[col], color=colours)
    a.set_title(title, fontsize=11); a.tick_params(axis="x", labelsize=8, rotation=15)
    lo, hi = opt_df[col].min(), opt_df[col].max()
    if col.startswith("Acc"):
        a.set_ylim(max(0, lo - 5), min(100, hi + 3))   # zoom in: the differences are small
    else:
        a.set_ylim(0, hi * 1.25)
    for k, v in enumerate(opt_df[col]):
        a.text(k, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/optimisation_comparison.png", bbox_inches="tight"); plt.show()
print(f"✔ Saved {OUT}/optimisation_comparison.png")

**Running the CNN less often.** The robot does not need a new label every 64 ms. Classifying one frame every *N* steps divides the CPU load by *N*:

In [ ]:
header("Step 7.4  CPU load when the CNN runs only every N control steps")
fast = opt_df.loc[opt_df["Time per image (ms)"].idxmin()]
show_table(pd.DataFrame([{"Classify every": f"{n} step{'s' if n > 1 else ''}", "Interval (ms)": n * 64,
                       "CPU load, FP32 (%)": round(base["Time per image (ms)"] / (n * 64) * 100, 2),
                       f"CPU load, {fast['Model']} (%)": round(fast["Time per image (ms)"] / (n * 64) * 100, 2)}
                      for n in (1, 4, 8, 16)]),
           "CPU load when the CNN runs every N control steps",
           fmt={"CPU load, FP32 (%)": "{:.2f}", f"CPU load, {fast['Model']} (%)": "{:.2f}"})
print("✔ Proposal for the robot: static int8 model + classify every 8 steps + run the CNN in a background thread")
info("The Webots controller (Assignment_3.py) uses the background thread and one CPU thread; set USE_INT8 = True for int8")

---
## 8. Save and download

* `cifar10_cnn.pth`: copy it into `controllers/Assignment_3/` in Webots for live predictions.
* `cnn_outputs.zip`: every figure and table from this notebook, for the slides and the report.

In [ ]:
header("Step 8.1  Saving the model, figures and results")
torch.save(fp32.state_dict(), "cifar10_cnn.pth")
info(f"cifar10_cnn.pth ({os.path.getsize('cifar10_cnn.pth')/1e6:.2f} MB): copy into controllers/Assignment_3/ in Webots")
results = {"test_accuracy_%": round(TEST_ACC * 100, 2), "parameters": TOTAL_PARAMS, "epochs": EPOCHS,
           "training_minutes": round(TRAIN_TIME / 60, 1),
           "per_class_accuracy_%": {c: round(float(a) * 100, 1) for c, a in zip(CLASSES, per_class)},
           "image_predictions": IMAGE_RESULTS, "optimisation": OPT_RESULTS}
json.dump(results, open(f"{OUT}/results.json", "w"), indent=2)
opt_df.to_csv(f"{OUT}/optimisation.csv", index=False)
!cp cifar10_cnn.pth cnn_model.py {OUT}/ && zip -qr cnn_outputs.zip {OUT}
for f in sorted(os.listdir(OUT)):
    info(f"{OUT}/{f}")
try:
    from google.colab import files
    files.download("cifar10_cnn.pth"); files.download("cnn_outputs.zip")
    print("✔ Downloading cifar10_cnn.pth and cnn_outputs.zip to your computer")
except ImportError:
    print("✔ Saved cifar10_cnn.pth and cnn_outputs.zip")

---
## 9. Summary, key findings and future work

In [ ]:
header("Step 9.1  Summary of the results")
summary_rows = [("Test accuracy", f"{TEST_ACC*100:.2f}% on {len(test_set):,} test images"),
                ("Model", f"{TOTAL_PARAMS:,} parameters, {EPOCHS} epochs, {TRAIN_TIME/60:.1f} min on {DEVICE.type.upper()}")]
summary_rows += [(r["Source"], f"{r['Prediction']} ({r['Confidence (%)']}%)") for r in IMAGE_RESULTS]
s8 = opt_df[opt_df.Model == "Static int8"].iloc[0]
summary_rows.append(("Best for the robot", f"Static int8: {s8['Accuracy (%)']}% accuracy, {s8['Time per image (ms)']} ms, "
                     f"{s8['Size (MB)']} MB ({s8['Speed-up']} faster, {s8['Smaller by']} smaller)"))
show_table(pd.DataFrame(summary_rows, columns=["Item", "Result"]), "Summary of the results")
print("✔ All steps finished")

### Key findings
* A compact VGG-style CNN (about 0.8 M parameters) reaches **about 88% test accuracy** on CIFAR-10 in a few minutes of GPU training, far better than a simple 3-layer CNN (about 70%).
* **Vehicles are recognised best**; **cat and dog** are confused most often because they look alike at 32 × 32 pixels.
* My own photos are classified correctly when the object fills the picture; the **Webots camera image** is harder (low resolution, rendered 3D model, low viewpoint).
* **Static int8 quantisation** makes the model about 4 times smaller and about 2 times faster on the CPU with almost no loss of accuracy, so the CNN fits easily inside the robot's 64 ms control step.

### Future work
1. **Fine-tune on Webots images**: collect labelled e-puck camera images of the animals and blocks, and fine-tune the CNN to close the gap between photos and simulation.
2. **Detect and classify blocks in Webots**: add the coloured blocks as extra classes, or use a small detector (for example SSD-MobileNet or YOLO-nano) to find *where* the object is in the image, so the robot can steer towards it.
3. **Lighter models**: structured pruning (removing whole filters) and a MobileNet-style network for real speed-ups on an embedded processor.
4. **Edge deployment**: export the int8 model to ONNX or TensorFlow Lite and run it on a microcontroller-class board on a real e-puck.

### References
* A. Krizhevsky, *Learning Multiple Layers of Features from Tiny Images*, University of Toronto, 2009 (CIFAR-10).
* K. Simonyan and A. Zisserman, *Very Deep Convolutional Networks for Large-Scale Image Recognition*, ICLR, 2015 (VGG).
* K. He, X. Zhang, S. Ren and J. Sun, *Deep Residual Learning for Image Recognition*, CVPR, 2016 (ResNet).
* S. Ioffe and C. Szegedy, *Batch Normalization*, ICML, 2015.
* B. Jacob et al., *Quantization and Training of Neural Networks for Efficient Integer-Arithmetic-Only Inference*, CVPR, 2018.
* Cyberbotics, *Webots R2023b documentation: e-puck*, https://cyberbotics.com/doc/guide/epuck